# Tutorial: GEMMES — Keen + Climate on the Pacioli Manifold

GEMMES (General Monetary and Multisectoral Macrodynamics for the Ecological Shift)
extends the Keen predator-prey model with a carbon cycle and climate damage function.
It is the canonical model for studying the interaction between private debt dynamics
and climate risk.

The state vector is now five-dimensional:

| Variable | Symbol | Meaning |
| --- | --- | --- |
| Wage share | ω | Labour's share of GDP |
| Employment rate | λ | Fraction of workforce employed |
| Debt ratio | d | Private debt / GDP |
| Temperature anomaly | ΔT | Global warming above pre-industrial (°C) |
| CO₂ concentration | CO₂ | Atmospheric carbon (GtC) |

The climate variables feed back into the economy via the **Nordhaus damage function**:

$$D(\Delta T) = 1 - \frac{1}{1 + \pi_1 \Delta T + \pi_2 \Delta T^2}$$

which reduces the effective profit share: $\pi_\text{net} = (1 - \omega - rd)(1 - D(\Delta T))$.

**What this tutorial adds beyond the Keen tutorial:**

1. Climate-augmented ODE with CO₂ and temperature dynamics
2. `CurvedBalanceSheet` — stranded asset risk as non-zero curvature field F
3. PCL `fold(β, [brown, green, deleverage])` — the three-way green transition switch
4. TIR routing: how a carbon tax shifts investment weights
5. 4-player thermal Shapley (workers / firms / banks / policy)

---

*Reference: Bovari, Giraud, McIsaac & Chancel (2018) Ecological Economics 147, 383–398.  
EconIAC implementation: Buckley (2026), doi:10.5281/zenodo.20262070*

In [ ]:
# Install econiac if running in Colab
# !pip install econiac jax[cpu]

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from econiac.core.manifold import (
    BalanceSheet, CurvedBalanceSheet, holonomy,
)
from econiac.pcl import flow, sequence, fold, conservation_loss, compile as pcl_compile
from econiac.routing import tir_from_scores, route
from econiac.routing.attribution import full_shapley_analysis

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. The GEMMES ODE system

We implement the five-variable system directly in JAX so it is differentiable
end-to-end.  The key additions over the Keen model are:

- **Nordhaus damage** $D(\Delta T)$ erodes the profit share
- **Radiative forcing** $F(\text{CO}_2) = F_{2\times} \log_2(\text{CO}_2 / \text{CO}_{2,\text{pre}})$ drives warming
- **Carbon cycle** $\dot{\text{CO}}_2 = \sigma(1-n)Y - \delta_{\text{CO}_2} \cdot \text{CO}_2$ where $n$ is the abatement fraction

In [ ]:
PARAMS = dict(
    alpha=0.02, beta=0.01, gamma=0.01, nu=3.0, r=0.03,
    phi0=-0.04, phi1=0.0006,
    kappa0=0.0, kappa1=0.025, kappa2=2.0,   # damped investment for stable orbit
    pi1=0.0, pi2=0.00236,                    # Nordhaus (2008) damage coefficients
    tau_T=50.0, lambda_T=1.13, F2x=3.7,
    CO2_pre=280.0, sigma=0.02, delta_CO2=0.01, n_tax=0.10,
)

def damage(T, p):
    return 1.0 - 1.0 / (1.0 + p["pi1"] * T + p["pi2"] * T**2)

def gemmes_ode(state, t, p):
    omega, lam, d, T, CO2 = state
    D     = damage(T, p)
    pi    = (1.0 - omega - p["r"] * d) * (1.0 - D)
    phi   = p["phi0"] + p["phi1"] / jnp.maximum(1.0 - lam, 0.01)**2
    kappa = (p["kappa0"] + p["kappa1"] * jnp.exp(p["kappa2"] * pi)) / p["nu"]
    F     = p["F2x"] * jnp.log(jnp.maximum(CO2, 1.0) / p["CO2_pre"]) / jnp.log(2.0)
    return jnp.array([
        omega * (phi - p["alpha"]),
        lam   * (kappa - p["alpha"] - p["beta"]),
        kappa - pi - (p["alpha"] + p["beta"]) * d,
        (F - p["lambda_T"] * T) / p["tau_T"],
        p["sigma"] * (1.0 - p["n_tax"]) * 100.0 - p["delta_CO2"] * CO2,
    ])

def simulate(omega0=0.78, lam0=0.95, d0=0.05, T0=0.85, CO2_0=395.0,
             T_end=80.0, dt=0.02, params=None):
    p = params or PARAMS
    state = jnp.array([omega0, lam0, d0, T0, CO2_0])
    n     = int(round(T_end / dt))
    times, traj = [0.0], [state]
    t = 0.0
    for _ in range(n):
        state = state + gemmes_ode(state, t, p) * dt
        t    += dt
        times.append(t); traj.append(state)
    return jnp.array(times), jnp.stack(traj)

times, traj = simulate()
omega, lam, d, T_anom, CO2 = [traj[:, i] for i in range(5)]

print(f"T = {float(times[-1]):.0f} years  |  {len(times)} steps")
print(f"Peak ΔT:   {float(T_anom.max()):.2f}°C  at t={float(times[int(T_anom.argmax())]):.0f}")
print(f"Peak debt: {float(d.max()):.3f}     at t={float(times[int(d.argmax())]):.0f}")
print(f"Damage at peak ΔT: {float(damage(T_anom.max(), PARAMS))*100:.2f}% of GDP")

In [ ]:
fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[1, 2])

t = times

ax0.plot(t, omega, color="steelblue"); ax0.set_title("Wage share ω"); ax0.set_xlabel("Years")
ax1.plot(t, lam,   color="seagreen");  ax1.set_title("Employment λ"); ax1.set_xlabel("Years")
ax2.plot(t, d,     color="crimson")
ax2.axvline(float(times[int(d.argmax())]), ls="--", color="grey", lw=0.8)
ax2.set_title("Debt ratio d"); ax2.set_xlabel("Years")

ax3.plot(t, T_anom, color="darkorange")
ax3.axhline(1.5, ls="--", color="red", lw=0.8, label="Paris 1.5°C")
ax3.axhline(2.0, ls="--", color="darkred", lw=0.8, label="Paris 2.0°C")
ax3.set_title("Temperature anomaly ΔT (°C)"); ax3.set_xlabel("Years"); ax3.legend(fontsize=8)

ax4.plot(t, CO2, color="saddlebrown")
ax4.axhline(280, ls="--", color="grey", lw=0.8, label="pre-industrial")
ax4.set_title("CO₂ (GtC)"); ax4.set_xlabel("Years"); ax4.legend(fontsize=8)

D_path = jnp.array([float(damage(T_anom[i], PARAMS)) * 100 for i in range(len(t))])
ax5.plot(t, D_path, color="purple")
ax5.set_title("Climate damage D(ΔT) (% of GDP)"); ax5.set_xlabel("Years")

fig.suptitle("GEMMES simulation: Keen debt dynamics + climate feedback", fontsize=13)
plt.show()

## 2. Stranded assets as curvature

At the Minsky-climate moment, brown capital is systematically overvalued.  The
market price of fossil-fuel assets embeds an implicit assumption that they will
continue to generate profits — but climate policy and physical damage will strand
some of them before they are fully utilised.

The **gap between the market price and the climate-adjusted price** is a genuine
arbitrage: it persists not because markets are irrational but because the risk
(policy reversal, physical damage, transition speed) has not yet been priced in.

In gauge-theory language: this is **non-zero curvature** on the Pacioli manifold.
The field strength tensor $F$ measures the arbitrage profit on a round trip through
brown and green capital markets.

$$\partial^2(bs) = F \neq 0$$

`CurvedBalanceSheet` carries this field strength explicitly, making it observable,
computable, and part of the calibration target.

In [ ]:
# Track the stranded asset premium (curvature ||F||) along the trajectory.
# Premium = climate damage × debt ratio — how much brown capital is overvalued
# relative to its climate-adjusted net present value.

peak_idx = int(d.argmax())
stranded_premiums = []
holonomies        = []

for i in range(0, len(times), 20):
    T_i   = float(T_anom[i])
    d_i   = float(d[i])
    om_i  = float(omega[i])
    D_i   = float(damage(jnp.array(T_i), PARAMS))
    premium = D_i * max(d_i, 0.0) * 50.0   # NPV of stranded loss on brown debt

    positions = jnp.array([
        [ om_i,  0.0,    0.0  ],
        [-om_i, -d_i,    0.2  ],
        [ 0.0,   d_i,   -0.2  ],
        [ 0.0,   premium, 0.0 ],
    ])
    F = jnp.array([0.0, premium, 0.0])

    cbs = CurvedBalanceSheet(
        positions=positions,
        sectors=["workers", "firms", "banks", "climate"],
        instruments=["wages", "brown_capital", "green_capital"],
        curvature=F,
    )
    stranded_premiums.append(float(jnp.linalg.norm(F)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(times[::20], stranded_premiums, color="saddlebrown", lw=2)
axes[0].set_xlabel("Years"); axes[0].set_ylabel("||F|| (stranded asset curvature)")
axes[0].set_title("Stranded asset risk (field strength ||F||)\nalong GEMMES trajectory")

axes[1].scatter(d[::20], T_anom[::20], c=times[::20], cmap="plasma", s=8)
axes[1].set_xlabel("Debt ratio d"); axes[1].set_ylabel("Temperature anomaly ΔT (°C)")
axes[1].set_title("Climate-debt phase portrait\ncolour = time")
cb = plt.colorbar(axes[1].collections[0], ax=axes[1])
cb.set_label("Years")

plt.tight_layout()
plt.show()

print(f"Peak stranded asset curvature ||F|| = {max(stranded_premiums):.4f}")
print(f"  (at t ≈ {float(times[::20][stranded_premiums.index(max(stranded_premiums))]):.0f})")